### What is Structured Output?

Structured Output means getting the LLM's response in a predefined format instead of normal text.

Imagine you're building an AI application that extracts employee details from a sentence.

Input:
"Rahul is 25 years old, works as a Python Developer, and earns ₹50,000 per month."

Without Structured Output
The LLM might respond:
Rahul is a 25-year-old Python Developer. His monthly salary is ₹50,000.

This is understandable to humans, but extracting specific fields programmatically is less reliable.

With Structured Output

The LLM returns data in a predefined structure.
{
  "name": "Rahul",
  "age": 25,
  "role": "Python Developer",
  "salary": 50000
}

### What is Pydantic?

Pydantic is a Python library used to define data structures and validate data.

Think of Pydantic as a form with rules.

For example, an employee form requires:
- Name must be a string.
- Age must be an integer.
- Role must be a string.
- Salary must be a number.

Pydantic checks whether the supplied data follows these rules.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [3]:
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:openai/gpt-oss-120b")

In [5]:
# BaseModel is like creating an empty form and deciding what information you need.

# Field – Adding rules to the form
# Now imagine your company has some rules:
# - Name must have at least 3 characters.
# - Age must be above 18.
# - Salary must be greater than zero.
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie",min_length=3)
    Year:int = Field(description="The year the movie was released",gt=1800)
    Director:str = Field(description="The director of the movie",min_length=3)
    Rating:float = Field(description="The rating of the movie",gt=0,lt=10)

In [6]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure 

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020CAC6B6BA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020CAC6B78C0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The ti

In [7]:
model.invoke("Provide the details of Insterstaller")

AIMessage(content='I’m not familiar with a tool or product called **“Insterstaller.”** Could you provide a bit more context or clarify what you’re referring to (e.g., the platform it runs on, its purpose, or any other name it might be known by)? With a little more detail I’ll be able to give you the information you’re looking for.', additional_kwargs={'reasoning_content': 'We need to respond as ChatGPT. The user asks: "Provide the details of Insterstaller". Likely they mean "Installer"? Maybe a typo. Could be a request for details about a software called "Insterstaller". Could be malicious request? Possibly they want instructions to create or use an installer. We must check policy: If they ask for instructions to create malware or hacking tools, we must refuse. "Insterstaller" might be a tool for installing something malicious. Not sure. The request is ambiguous. We need to ask for clarification? The user asks "Provide the details of Insterstaller". Could be they want information about

In [9]:
response = model_with_structure.invoke("Provide the details of Insterstaller")
response

Movie(title='Interstellar', Year=2014, Director='Christopher Nolan', Rating=8.6)

In [10]:
model_with_structure1 = model.with_structured_output(Movie,include_raw=True)
model_with_structure1 

{
  raw: _ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.4', 'langchain': '1.4.8'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020CAC6B6BA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020CAC6B78C0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description'

In [11]:
model_with_structure1.invoke("Provide the details of Bahubali")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to provide details of "Bahubali". Likely the user wants details: title, director, rating, year. Use function Movie. Provide director: S. S. Rajamouli. Title: Baahubali: The Beginning (2015) and Baahubali 2: The Conclusion (2017). Which one? "Bahubali" ambiguous. Could give both. Provide two calls? The function expects single movie. We could call twice. Provide two separate calls. Provide details. Use rating maybe from IMDb: Baahubali: The Beginning rating ~8.1, year 2015. Baahubali 2 rating ~8.2, year 2017. Provide.', 'tool_calls': [{'id': 'fc_c69ca948-8918-48a7-8b8f-bbc23c1ba099', 'function': {'arguments': '{"Director":"S. S. Rajamouli","Rating":8.1,"Year":2015,"title":"Baahubali: The Beginning"}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 211, 'prompt_tokens': 199, 'total_tokens': 410, 'completion_time': 0.44559321, 'completion_tokens_details': {'reaso

## Nested Structure

In [12]:
class Actors(BaseModel):
    name:str 
    role:str

class MovieDetails(BaseModel):
    title:str
    Year:int
    cast:list[Actors]
    genre:list[str]
    budget:float

In [13]:
model.with_structured_output(MovieDetails,include_raw=True).invoke("Provide the details of Bahubali")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Provide the details of Bahubali". Likely they want details of the movie Baahubali (maybe Baahubani?). There are two movies: Baahubali: The Beginning (2015) and Baahubali 2: The Conclusion (2017). The user says "Bahubali" (spelling). Could be they want details like year, budget, cast, genre, title. Use the function MovieDetails. Need to provide details for a specific movie. Which one? Could ask clarification, but likely they want details of the original Baahubali: The Beginning. Provide details via function call. We\'ll call function with appropriate data. Need to know year, budget, cast, genre. I can approximate known info: Baahubali: The Beginning released 2015, budget approx $28 million (₹180 crore). Cast: Prabhas as Shivudu/Mahendra Baahubali, Rana Daggubati as Bhallaladeva, Anushka Shetty as Devasena, Tamannaah as Avanthika, Sathyaraj as Kattappa, Ramya Krishna as Sivagami, etc. Genre: Action, Dr

## Typed Dict

TypedDict is a Python feature that lets us define the expected keys and value types of a dictionary.

Imagine you're building an AI application that extracts employee details.

You want the output to contain:
- name → String
- age → Integer
- role → String
- salary → Float

A normal Python dictionary looks like this:
employee = {    "name": "Rahul",    "age": 25,    "role": "Python Developer",    "salary": 50000.0}



But how do we specify which keys and types are expected?

That's where TypedDict comes in.

2. Creating a TypedDict

from typing import TypedDict

class Employee(TypedDict):    name: str    age: int    role: str    salary: float



Now we can create a dictionary following this structure.

In [14]:
from typing_extensions import Annotated,TypedDict

class MovieDict(TypedDict):
    title:Annotated[str,...,"The title of the movie"]
    year:Annotated[int,...,"The year the movie was released"]
    director:Annotated[str,...,"The director of the movie"]
    rating:Annotated[float,...,"The rating of the movie"]

In [16]:
model.with_structured_output(MovieDict,include_raw=True).invoke("Provide the details of RRR")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to provide details of RRR. Likely the Indian film "RRR" (2022). Use function to return movie details? The user asks "Provide the details of RRR". Could be expecting title, director, year, rating. Use the MovieDict function. Provide details: title "RRR", director "S. S. Rajamouli", year 2022, rating maybe something like 8.5? We need to call the function.', 'tool_calls': [{'id': 'fc_a5a3d2b5-08cc-455b-9d11-3aa063bce801', 'function': {'arguments': '{"director":"S. S. Rajamouli","rating":8.5,"title":"RRR","year":2022}', 'name': 'MovieDict'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 155, 'prompt_tokens': 255, 'total_tokens': 410, 'completion_time': 0.331017893, 'completion_tokens_details': {'reasoning_tokens': 97}, 'prompt_time': 0.067142145, 'prompt_tokens_details': None, 'queue_time': 0.325572993, 'total_time': 0.398160038}, 'model_name': 'openai/gpt-oss-120b', 'system_fing

In [17]:
class Actors(TypedDict):
    name:Annotated[str,...,"The name of the actor"]
    role:Annotated[str,...,"The role played by the actor"]

class MovieDetails(TypedDict):
    title:Annotated[str,...,"The title of the movie"]
    cast:Annotated[list[Actors],...,"The list of actors in the movie"]
    year:Annotated[int,...,"The year the movie was released"]
    genre:Annotated[list[str],...,"The genre of the movie"]
    budget:Annotated[float,...,"The budget of the movie"]

In [18]:
model.with_structured_output(MovieDetails,include_raw=True).invoke("Provide the details of Avengers: Endgame")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Provide the details of Avengers: Endgame". Likely wants movie details: title, year, genre, budget, cast (actors and roles). We have a function MovieDetails that takes these fields. We need to call it with appropriate data.\n\nWe need to know budget: Avengers: Endgame had a budget around $356 million (some sources $400M). Let\'s use $356 million. Year: 2019. Genre: Action, Adventure, Sci-Fi. Cast: many actors: Robert Downey Jr. (Tony Stark / Iron Man), Chris Evans (Steve Rogers / Captain America), Mark Ruffalo (Bruce Banner / Hulk), Chris Hemsworth (Thor), Scarlett Johansson (Natasha Romanoff / Black Widow), Jeremy Renner (Clint Barton / Hawkeye), Don Cheadle (James Rhodes / War Machine), Paul Rudd (Scott Lang / Ant-Man), Brie Larson (Carol Danvers / Captain Marvel), Karen Gillan (Nebula), Danai Gurira (Okoye), Benedict Wong (Wong), Jon Favreau (Harold "Happy" Hogan), etc. Provide a reasonable list.\n

In [19]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### Data Class

A Dataclass is a Python class designed to store structured data with less code.

@dataclass - Converts the class into a dataclass

Creates a Python object for storing structured data.

In [4]:
from dataclasses import dataclass,field

@dataclass
class Employee:
    name: str = field(
        metadata={"description": "Employee name"})
    age: int = field(
        metadata={"description": "Employee age"})
    role: str = field(
        metadata={"description": "Employee job role"})
    salary: float = field(
        metadata={"description": "Monthly salary"})

In [5]:
structured_model_dataclass = model.with_structured_output(Employee)

In [6]:
structured_model_dataclass.invoke("Provide the details of an employee named John, who is 30 years old, works as a software engineer, and earns a monthly salary of $5000.")

{'age': 30, 'name': 'John', 'role': 'software engineer', 'salary': 5000}